# 🎬 Prédiction du Genre des Films avec Naive Bayes

---

**Objectif** : Prédire le genre d'un film (Action, Drame, Comédie, etc.) en fonction de ses caractéristiques numériques à l'aide d'un classificateur **Naive Bayes**.

**Variables utilisées** :
- `Votes` : Le nombre de votes reçus par le film
- `Popularity` : Un score de popularité calculé à partir des votes
- `Rating` : La note moyenne du film

**Variable cible** : `Genre` du film

**Source des données** : [IMDb Datasets](https://datasets.imdbws.com/)

---

# I. Importation des bibliothèques

In [ ]:
# Manipulation de données
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Prétraitement
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Modélisation
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

# Évaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    precision_score
)

# Configuration
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

✅ Toutes les bibliothèques ont été importées avec succès.


---
# II. Chargement et préparation des données

On utilise deux fichiers du dataset IMDb :
- **`title.basics.tsv.gz`** → informations générales (titre, type, année, genres)
- **`title.ratings.tsv.gz`** → notes et votes des utilisateurs

Ces deux fichiers sont reliés par la colonne `tconst` (identifiant unique de chaque titre).

## 2.1 — Chargement des fichiers bruts

In [5]:
# Chargement de title.basics
basics = pd.read_csv('data/title.basics.tsv', sep='\t', low_memory=False, na_values='\\N')
print(f"title.basics : {basics.shape[0]:,} lignes × {basics.shape[1]} colonnes")
print(f"Colonnes : {list(basics.columns)}\n")

# Chargement de title.ratings
ratings = pd.read_csv('data/title.ratings.tsv', sep='\t', low_memory=False, na_values='\\N')
print(f"title.ratings : {ratings.shape[0]:,} lignes × {ratings.shape[1]} colonnes")
print(f"Colonnes : {list(ratings.columns)}")

title.basics : 12,376,284 lignes × 9 colonnes
Colonnes : ['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genres']

title.ratings : 1,650,900 lignes × 3 colonnes
Colonnes : ['tconst', 'averageRating', 'numVotes']


## 2.2 — Fusion des deux datasets

In [8]:
# Jointure interne sur tconst
df = pd.merge(basics, ratings, on='tconst', how='inner')
print(f"Dataset fusionné : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head()

Dataset fusionné : 1,650,900 lignes × 11 colonnes


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1,"Documentary,Short",5.7,2200
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5,"Animation,Short",5.5,312
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5,"Animation,Comedy,Romance",6.4,2308
3,tt0000004,short,Un bon bock,Un bon bock,0,1892.0,NaN,12,"Animation,Short",5.1,196
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893.0,NaN,1,Short,6.2,3037


## 2.3 — Filtrage : garder uniquement les films

Le dataset IMDb contient plusieurs types de contenus (séries, courts-métrages, épisodes TV, etc.). On ne conserve que les **films** (`titleType == 'movie'`).

In [ ]:
# Filtrer les films uniquement
df = df[df['titleType'] == 'movie'].copy()
print(f"✅ Nombre de films retenus : {df.shape[0]:,}")

## 2.4 — Traitement de la colonne genres

Dans IMDb, un film peut être associé à **plusieurs genres** séparés par des virgules (ex : `"Action,Drama,Thriller"`). Pour simplifier la classification, on extrait uniquement le **genre principal** (le premier listé).

In [ ]:
# Supprimer les films sans genre renseigné
df = df.dropna(subset=['genres'])
print(f"Films avec genre renseigné : {df.shape[0]:,}")

# Extraire le genre principal (premier de la liste)
df['Genre'] = df['genres'].str.split(',').str[0]

# Distribution complète
print(f"\nNombre de genres uniques : {df['Genre'].nunique()}")
print(f"\nDistribution de tous les genres :")
print(df['Genre'].value_counts())

In [ ]:
# Garder uniquement les genres avec suffisamment de films (>= 500)
# pour que le modèle ait assez de données par classe
genre_counts = df['Genre'].value_counts()
genres_to_keep = genre_counts[genre_counts >= 500].index.tolist()

print(f"Genres retenus (>= 500 films) : {genres_to_keep}")

df = df[df['Genre'].isin(genres_to_keep)].copy()
print(f"\n✅ Nombre de films après filtrage des genres : {df.shape[0]:,}")
print(f"\nDistribution finale :")
print(df['Genre'].value_counts())

## 2.5 — Création de la variable Popularity

La popularité n'existe pas directement dans les données IMDb. On la calcule en normalisant le nombre de votes sur une **échelle logarithmique** (entre 0 et 100), car la distribution des votes suit une loi de puissance.

In [ ]:
# Calcul du score de popularité (0-100, échelle log)
df['Popularity'] = np.log1p(df['numVotes'])
df['Popularity'] = (
    (df['Popularity'] - df['Popularity'].min()) /
    (df['Popularity'].max() - df['Popularity'].min()) * 100
).round(1)

print("✅ Score de popularité créé (0–100) :")
print(df['Popularity'].describe().round(2))

## 2.6 — Sélection et renommage des colonnes

In [ ]:
# Renommage pour plus de clarté
df = df.rename(columns={
    'primaryTitle': 'Title',
    'numVotes': 'Votes',
    'averageRating': 'Rating'
})

# Sélection des colonnes utiles
df = df[['Title', 'Votes', 'Popularity', 'Rating', 'Genre']].copy()
df = df.reset_index(drop=True)

print(f"✅ Dataset final : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head(10)

## 2.7 — Nettoyage des valeurs manquantes

In [ ]:
# Vérification des valeurs manquantes
print("Valeurs manquantes par colonne :")
print(df.isnull().sum())

# Suppression des lignes incomplètes
df = df.dropna()
print(f"\n✅ Dataset après nettoyage : {df.shape[0]:,} lignes")

---
# IV. Préparation des données pour la modélisation

## 4.1 — Encodage de la variable cible (Genre)

In [ ]:
# Encodage des genres en valeurs numériques
le = LabelEncoder()
df['Genre_encoded'] = le.fit_transform(df['Genre'])

# Afficher la correspondance
print("Correspondance Genre → Code numérique :")
print("-" * 40)
for genre, code in zip(le.classes_, range(len(le.classes_))):
    nb = (df['Genre_encoded'] == code).sum()
    print(f"  {genre:15s} → {code}  ({nb:,} films)")

## 4.2 — Définition des variables X (features) et y (cible)

In [ ]:
# Variables indépendantes
X = df[['Votes', 'Popularity', 'Rating']]

# Variable cible
y = df['Genre_encoded']

print(f"Dimensions de X (features) : {X.shape}")
print(f"Dimensions de y (cible)    : {y.shape}")
print(f"\nFeatures utilisées : {list(X.columns)}")
print(f"Classes cibles     : {list(le.classes_)}")
X.head()

## 4.3 — Séparation en ensemble d'entraînement et de test

In [ ]:
# Séparation 80% entraînement / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Conserver la proportion des classes
)

print(f"Ensemble d'entraînement : {X_train.shape[0]:,} films ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Ensemble de test        : {X_test.shape[0]:,} films ({X_test.shape[0]/len(X)*100:.0f}%)")

# Vérifier la stratification
print(f"\nDistribution dans l'ensemble d'entraînement :")
print(pd.Series(y_train).value_counts().sort_index().rename(dict(enumerate(le.classes_))))

print(f"\nDistribution dans l'ensemble de test :")
print(pd.Series(y_test).value_counts().sort_index().rename(dict(enumerate(le.classes_))))

## 4.4 — Normalisation des features

Le Naive Bayes gaussien fonctionne mieux quand les variables suivent des distributions proches de la normale. On applique un **StandardScaler** pour centrer et réduire les features.

In [ ]:
# Normalisation (centrer-réduire)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Vérification
print("Moyennes après normalisation (train) :", X_train_scaled.mean(axis=0).round(4))
print("Écarts-types après normalisation (train) :", X_train_scaled.std(axis=0).round(4))
print("\n✅ Données normalisées.")

---
# V. Entraînement du modèle Naive Bayes

## 5.1 — Création et entraînement du modèle

On utilise le **GaussianNB** (Naive Bayes Gaussien), adapté aux variables continues. Ce modèle suppose que chaque feature suit une **distribution normale** au sein de chaque classe.

In [ ]:
# Création du modèle
model = GaussianNB()

# Entraînement sur les données d'entraînement
model.fit(X_train_scaled, y_train)

print("✅ Modèle Naive Bayes entraîné avec succès !")
print(f"\nClasses apprises : {list(le.classes_)}")
print(f"Nombre de features : {model.n_features_in_}")

## 5.2 — Prédictions sur l'ensemble de test

In [ ]:
# Prédictions
y_pred = model.predict(X_test_scaled)

# Probabilités de prédiction
y_pred_proba = model.predict_proba(X_test_scaled)

# Aperçu des résultats
resultats = pd.DataFrame({
    'Genre réel': le.inverse_transform(y_test),
    'Genre prédit': le.inverse_transform(y_pred),
    'Correct': y_test.values == y_pred
})

print("Aperçu des 15 premières prédictions :")
resultats.head(15)

---
# VI. Évaluation des performances

## 6.1 — Métriques globales

In [ ]:
# Calcul des métriques
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print("=" * 50)
print("       MÉTRIQUES DE PERFORMANCE GLOBALES")
print("=" * 50)
print(f"  Accuracy  (Exactitude)  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  Precision (Précision)   : {precision:.4f}")
print(f"  Recall    (Rappel)      : {recall:.4f}")
print(f"  F1-Score                : {f1:.4f}")
print("=" * 50)

## 6.2 — Rapport de classification détaillé par genre

In [ ]:
# Rapport complet par classe
print("Rapport de classification par genre :")
print("=" * 65)
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 6.3 — Matrice de confusion

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=1, linecolor='white',
            annot_kws={'size': 14, 'fontweight': 'bold'},
            ax=ax)
ax.set_title('Matrice de Confusion', fontsize=16, fontweight='bold')
ax.set_xlabel('Genre prédit', fontsize=13)
ax.set_ylabel('Genre réel', fontsize=13)
plt.tight_layout()
plt.show()

---
# VII. Analyse des résultats

## 7.1 — Performance par genre (visualisation)

In [ ]:
# Extraire les métriques par classe depuis le classification_report
report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)

# Créer un DataFrame des performances par genre
perf_df = pd.DataFrame(report).T
perf_df = perf_df.loc[le.classes_]  # Garder uniquement les genres

# Graphique en barres groupées
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(le.classes_))
width = 0.25

bars1 = ax.bar(x - width, perf_df['precision'], width, label='Précision', color='#3498db', edgecolor='black')
bars2 = ax.bar(x,         perf_df['recall'],    width, label='Rappel',     color='#e74c3c', edgecolor='black')
bars3 = ax.bar(x + width, perf_df['f1-score'],  width, label='F1-Score',   color='#2ecc71', edgecolor='black')

ax.set_xlabel('Genre', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Précision, Rappel et F1-Score par Genre', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(le.classes_, rotation=45, ha='right')
ax.legend(fontsize=12)
ax.set_ylim(0, 1.05)
ax.axhline(y=accuracy, color='gray', linestyle='--', alpha=0.7, label=f'Accuracy globale ({accuracy:.2f})')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

---
# VIII. Conclusion et interprétation

### Résumé des résultats

### Interprétation

**Forces du modèle :**
- Le Naive Bayes est rapide à entraîner et donne des résultats interprétables.
- Les probabilités de prédiction permettent de mesurer la confiance du modèle.

**Limites observées :**
- Le modèle repose sur seulement **3 features numériques** (Votes, Popularity, Rating), ce qui limite la capacité de discrimination entre genres.
- L'hypothèse d'**indépendance** des features (propre au Naive Bayes) n'est pas parfaitement respectée : Votes et Popularity sont fortement corrélés.
- Certains genres partagent des profils statistiques très proches, rendant la classification difficile.

**Pistes d'amélioration :**
- Ajouter d'autres features (durée du film, année de sortie, budget, acteurs...).
- Tester d'autres algorithmes (Random Forest, SVM, KNN).
- Utiliser des techniques de NLP sur les synopsis pour enrichir les features.
- Effectuer une analyse en composantes principales (PCA) pour réduire la corrélation entre features.